<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week8_Day4_Daily_Challenge_MCP_Airbnb_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [3]:
!pip install -q mcp nest_asyncio requests
!pip install azure-ai-inference

# Optional: real Airbnb server
!npm install -g @openbnb/mcp-server-airbnb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 14.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼npm warn deprecated whatwg-encoding@3.1.1: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠼⠴⠦npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 124 packages in 10s
⠸
⠸51 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [4]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [5]:

import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = True  # True if npm server available
USE_REAL_LLM = True     # True if GITHUB_TOKEN set


In [4]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [6]:
import os
from google.colab import userdata  # Colab secrets API

# If your secret is saved under the key "GITHUB_TOKEN" in Colab:
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

# If you used a different key name in the secrets UI, e.g. "github_token":
# os.environ["GITHUB_TOKEN"] = userdata.get("github_token")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


GITHUB_TOKEN visible to Python: True


## Local notes MCP server

In [24]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("LocalNotes")
notes = []

@mcp.tool()
def add_note(text: str) -> str:
    """Add a note to the in-memory list."""
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    """List saved notes."""
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run(transport='stdio')
'''.strip(),
    encoding="utf-8",
)
print("Fichier local_notes_server.py corrigé (syntaxe fixée) et prêt.")

Fichier local_notes_server.py corrigé (syntaxe fixée) et prêt.


## Client helpers (convert tools, stub planner, optional real LLM)

In [11]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }



def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant with access to tools."},
            {"role": "user", "content": prompt}
        ],
        tools=functions,
        tool_choice="auto"
    )

    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [12]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [22]:
import sys
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def orchestrate(prompt: str):
    env = BASE_ENV.copy()
    env["PYTHONPATH"] = os.getcwd()

    # Run the script directly with python instead of 'mcp run'
    local_params = StdioServerParameters(
        command=sys.executable,
        args=[str(LOCAL_SERVER)],
        env=env,
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=env,
        )

    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)

                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        prefix, tool_name = name.split("__", 1)

                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                            tool_results.append({
                                "name": name,
                                "args": args,
                                "content": [c.text for c in res.content if hasattr(c, "text")]
                            })
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)
                            payload = [c.text for c in res.content if hasattr(c, "text")]
                            tool_results.append({
                                "name": name,
                                "args": args,
                                "content": payload
                            })

                    return tool_calls, tool_results

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [25]:
import asyncio
from IPython.display import Markdown
import sys
import importlib

prompt = "Quels outils as-tu à ta disposition ? Liste-les s'il te plaît."

try:
    # Diagnostic rapide avant de lancer
    print("Vérification du fichier serveur...")
    try:
        import local_notes_server
        importlib.reload(local_notes_server)
        print("Import réussi, lancement de l'orchestration...")
    except Exception as e_import:
        print(f"Erreur critique lors de l'importation du serveur : {e_import}")
        raise

    # Exécution avec un timeout
    tool_calls, tool_results = asyncio.run(asyncio.wait_for(orchestrate(prompt), timeout=60))

    print("Génération de la réponse finale...")
    final_answer = answer_with_llm(prompt, tool_calls, tool_results, use_real=USE_REAL_LLM)

    print("--- RÉPONSE FINALE ---")
    display(Markdown(final_answer))
except Exception as e:
    print(f"Erreur détectée : {type(e).__name__}: {e}")
    if "Connection closed" in str(e):
        print("\nLe serveur s'est déconnecté. Cela peut arriver si les flux stdio sont pollués par des prints.")
    import traceback
    traceback.print_exc()

Vérification du fichier serveur...
Import réussi, lancement de l'orchestration...
Génération de la réponse finale...
--- RÉPONSE FINALE ---


Je ne peux pas répondre directement à ta question car je n'ai pas effectué d'appel à un outil pour obtenir cette information. Si tu veux, je peux essayer de trouver une réponse en utilisant les outils disponibles.

## Tools used
- Aucun outil utilisé.